# Manfucatring Learning Curve Example

In [ ]:
!pip install -e "OneDrive/Documents/Projects/CaSES/PyCostTools"

ERROR: OneDrive/Documents/Projects/CaSES/PyCostTools is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).


In [ ]:
import pandas as pd
import numpy as np
from pycost.learn import lc_prep, lc


from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV
from pycost.analysis.constrained.constrained_model import ConstrainedRegression, ConstrainedRegressionCV
from pycost.analysis.model import Model, Models



import warnings
warnings.filterwarnings('ignore')

# Create manufacturing lot data for 8 lots with columns:
# - Lot Type (EMD, LRIP, FRP)
# - FY (Fiscal Year)
# - LotQuantity

# Create the lot data
def create_lot_data(number_of_lots=8, total_quantity=100, lrip_percentage=0.10, T1=100, LC=0.95, RC=0.95, noise=0.05):
    '''Create Fake Lot Data for Testing that is typical of what we see in the real world'''
   
    emd_quantity = 2
    lrip_lots = 3
    frp_lots = number_of_lots - lrip_lots-1

    lrip_quantity = np.round((total_quantity-emd_quantity) * lrip_percentage, 0)
    #print(lrip_quantity)
    lrip =[]
    cum_lrip = 0
    lrip= np.linspace(2, lrip_quantity, lrip_lots)
    #for i, lot in enumerate(lrip):
    #    lot_quantity = np.round(lot, 0)
    #    print("lot:", i, "quantity:", lot_quantity)
    #    if i == lrip_lots-1:
    #        lot_quantity = lrip_quantity-cum_lrip
    #    lrip[i]=lot_quantity
    #    cum_lrip += lot_quantity

    frp_quantity = np.round(total_quantity- lrip_quantity-emd_quantity, 0)

    #print(emd_quantity, lrip_quantity, frp_quantity, sum([emd_quantity, lrip_quantity, frp_quantity]))

    # lrip should increase each year for the first 3 years
    #lrip = [np.round(lrip_quantity/lrip_lots-lrip_lots, 1)]*lrip_lots
    
    # frp should incremenet by 1 each year
    frp_beg = np.round(frp_quantity/frp_lots-frp_lots, 0)
    frp_end = np.round(frp_quantity/frp_lots, 0)
    frp = np.arange(int(frp_beg), int(frp_end), 1)
    
    
    quantity = [emd_quantity] + list(lrip) + list(frp)
    fy = np.arange(2020, 2020+number_of_lots, 1)
    lot_type = ['EMD'] + ['LRIP']*lrip_lots + ['FRP']*frp_lots

    #print(len(quantity), len(fy), len(lot_type))
    #print(quantity, fy, lot_type)
    df = pd.DataFrame(dict(
        program=['program x']*number_of_lots,
        quantity = quantity,
        fy = fy,
        lot_type = lot_type
    ))

    df = lc_prep(df, cols=['program', 'fy'], val='quantity', lc_slope=0.95)
    #print(df)
    true_T1, true_b, true_c = T1, np.log(LC)/np.log(2), np.log(RC)/np.log(2)
    df["cost"] = true_T1 * df.midpoint**true_b * df.quantity**true_c * np.random.normal(1, noise, number_of_lots)

    return df


ModuleNotFoundError: No module named 'bokeh.models.arrow_heads'

In [ ]:
# Print the original data
df = pd.concat([create_lot_data(number_of_lots=5,lrip_percentage=.5, total_quantity=50), create_lot_data().assign(program='program y')])
df

,program,quantity,fy,lot_type,First,Last,midpoint,share_qty,cost
0,program x,2.0,2020,EMD,1.0,2.0,1.457107,2.0,86.505232
1,program x,2.0,2021,LRIP,3.0,4.0,3.482051,2.0,83.319491
2,program x,13.0,2022,LRIP,5.0,17.0,10.109772,13.0,73.850822
3,program x,24.0,2023,LRIP,18.0,41.0,28.333078,24.0,66.574579
4,program x,23.0,2024,FRP,42.0,64.0,52.422963,23.0,56.997078
0,program y,2.0,2020,EMD,1.0,2.0,1.457107,2.0,90.479903
1,program y,2.0,2021,LRIP,3.0,4.0,3.482051,2.0,93.779623
2,program y,6.0,2022,LRIP,5.0,10.0,7.285534,6.0,79.705667
3,program y,10.0,2023,LRIP,11.0,20.0,15.166198,10.0,71.152314
4,program y,18.0,2024,FRP,21.0,38.0,28.874447,18.0,61.799309
